# Playing with Sparsity

What we are doing here is trying to see whether we can reasonably hope to store sparse tensors containing detailed incidence structures for a large hypergraph. In this case, we'll store a 3D tensor `S` where `S[i,j,k] = 1` if nodes `i` and `j` are both in hyperedge `k`, and zero otherwise. This tensor has shape `(m, m, n)` where `m` is the number of hyperedges and `n` is the number of nodes.

In [1]:
import xgi
import torch
import sys

In [2]:
H = xgi.load_xgi_data("senate-bills")

n = len(H.nodes)
m = len(H.edges)

Here's our main loop to construct a list of 3-indices where the tensor should have nonzero entries. Note that there's a hardcoded limit of checking 1% of edges (like we might do in a batch of stochastic gradient descent), which would need to be removed for a full test. 

In [ ]:
batch_size = int(m / 100)

ix = []

for i, e in enumerate(H.edges): 
    
    # hardcoded in for testing
    if i >= batch_size: 
        break 
    
    if i % 1000 == 0:
        print(f"Processing edge {i} of {m}")
    # neighbors = H.edges.neighbors(e)
    
    for f in H.edges: 
        if f == e: 
            continue
        
        intersection = set(H.edges.members(e)) & set(H.edges.members(f))
        
        if len(intersection) != 0: 
            for v in intersection: 
                ix.append((int(e), int(f), int(v)))

Processing edge 0 of 29157
Processing edge 1000 of 29157


Having done the above, we now have a list of indices of the form `(e, f, v)` where `e` and `f` are hyperedge indices and v is a node index. We would *like* to form a 3d sparse tensor with shape `(m, m, n)` to hold these indices, but for some reason PyTorch does not support sparse tensors of more than 2 dimensions.

So, we need to do something kinda dumb: instead of a (m, m, n) tensor, we will make a (m, m*n) tensor, where the second index is formed by "flattening" the (e, f) pair into a single index of the form `e*m+f`.

In [4]:
# new second index is f*n + v, with mn total entries
# new shape is (m, mn)

wrapped_ix = [(e*m+f, v) for (e, f, v) in ix]

Now we are ready to make our dumb lil sparse tensor. 

In [5]:
s = torch.sparse_coo_tensor(
    indices=torch.tensor(wrapped_ix).T,
    values=torch.ones(len(wrapped_ix)),
    size=(m*m, n), check_invariants=False, requires_grad=False
)

Here's code to check the size of this tensor on disk, in MB: 

In [6]:
s.element_size()*s._nnz() / 1e6  # in MB

30.678352

That's 1% the size we'd need, so multiply by 100 to get an estimate of the size of the full sparse tensor we need to form. 

The point of doing this whole thing is that now we can form counts of things like "estimate of number of nodes of size 1 contained in the `e`-`f` pair" by matrix multiplication: 

In [7]:
z = torch.randn((n,), requires_grad=True) # some random labels

counts = s@z # e-f entry of counts is sum over v of s[e,f,v] * z[v], which is an estimate of the number of nodes in edge e and f of label 1

Now we can use cout for whatever we need next in our pipeline. Here's a quick example just to check that we can backprop to get gradients in $z$. 

In [ ]:
loss = (counts**2).mean()
loss.backward()

The thing that appears to be very cool about this is that the forward and backward step here are not exactly fast, but they *appear* to scale very well, sublinearly with batch size. So, although there's a lot of stuff here which is quite slow, once we have formed the relevant sparse tensors, we should be able to move through even large batches very quickly. 

In [ ]:
z.grad

tensor([ 0.0000e+00, -6.8998e-06,  6.8956e-06, -4.1993e-07, -1.8815e-03,
        -3.7423e-04,  2.3256e-03, -1.0661e-04,  6.3707e-06,  1.0089e-05,
         2.6314e-05,  3.4778e-05,  1.4836e-05,  5.5308e-06,  1.4337e-04,
         1.0103e-05, -2.1313e-04, -3.6785e-04, -2.6991e-04, -9.1422e-05,
         4.0392e-04, -2.1265e-05, -5.6796e-04,  9.0630e-05, -2.2619e-03,
         8.8544e-06,  3.4296e-04,  4.3410e-05,  2.4928e-04,  1.3502e-04,
        -1.8397e-04, -1.4696e-04,  2.1557e-06,  2.7532e-05, -1.9704e-04,
        -2.2687e-05, -6.9497e-04,  1.0816e-04, -4.4963e-04,  3.5709e-07,
        -8.8304e-05,  4.5303e-04,  1.2712e-05, -1.9563e-04, -9.6834e-05,
         1.4741e-04,  7.0074e-05,  5.5344e-05,  1.0874e-05,  1.0695e-05,
        -2.5695e-06,  4.3153e-05, -3.4509e-04,  9.6409e-05,  2.3903e-05,
        -2.8794e-05,  5.5494e-05,  3.2970e-04, -1.7145e-05,  2.1699e-04,
         1.9336e-04, -7.0360e-06, -3.9261e-05,  4.1693e-05,  4.1807e-05,
        -2.9358e-06, -1.1682e-04,  2.2150e-05, -3.1

Sweet, delicious gradients! 